# Linear Regression using pyspark

In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Practise').getOrCreate()

In [3]:
df = spark.read.csv("../datasets/sample_data_lr.csv", header = True, inferSchema=True)

In [4]:
df.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



- we try to predict the salary based on the Name and age

In [5]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [6]:
df.columns

['Name', 'age', 'Experience', 'Salary']

- unlike in traditional machine learning approach in sklearn, we don't split the data intro train test
- here we group the independent features as a new feature
- eg. ['Age','Experience'] ----> new feature ----> independent feature

In [7]:
from pyspark.ml.feature import VectorAssembler
featureassembler = VectorAssembler(inputCols=['age','Experience'], outputCol= 'Independent_features')

In [8]:
output = featureassembler.transform(df)

In [9]:
output.show()

+---------+---+----------+------+--------------------+
|     Name|age|Experience|Salary|Independent_features|
+---------+---+----------+------+--------------------+
|    Krish| 31|        10| 30000|         [31.0,10.0]|
|Sudhanshu| 30|         8| 25000|          [30.0,8.0]|
|    Sunny| 29|         4| 20000|          [29.0,4.0]|
|     Paul| 24|         3| 20000|          [24.0,3.0]|
|   Harsha| 21|         1| 15000|          [21.0,1.0]|
|  Shubham| 23|         2| 18000|          [23.0,2.0]|
+---------+---+----------+------+--------------------+



In [10]:
finalized_data = output.select("Independent_features", 'Salary')

In [11]:
finalized_data.show()

+--------------------+------+
|Independent_features|Salary|
+--------------------+------+
|         [31.0,10.0]| 30000|
|          [30.0,8.0]| 25000|
|          [29.0,4.0]| 20000|
|          [24.0,3.0]| 20000|
|          [21.0,1.0]| 15000|
|          [23.0,2.0]| 18000|
+--------------------+------+



In [12]:
from pyspark.ml.regression import LinearRegression

## Train test split 
train_data, test_data = finalized_data.randomSplit([0.75, 0.25])
## Train data --> 75% and test data --> 25%
regressor = LinearRegression(featuresCol = 'Independent_features', labelCol='Salary')
regressor = regressor.fit(train_data)

In [13]:
regressor.coefficients

DenseVector([-64.8464, 1584.7554])

In [14]:
regressor.intercept

15414.10693970376

### Prediction

In [15]:
pred_results = regressor.evaluate(test_data)

pred_results.predictions.show()

+--------------------+------+------------------+
|Independent_features|Salary|        prediction|
+--------------------+------+------------------+
|          [24.0,3.0]| 20000|18612.059158134223|
+--------------------+------+------------------+



In [16]:
pred_results.meanAbsoluteError, pred_results.meanSquaredError

(1387.9408418657767, 1926379.780519081)